# GPT OSS 120B — RLVR with Tool-Integrated Reasoning

Train `openai/gpt-oss-120b` via **RLVR** with **Python tool use** using the Tinker `tinker_cookbook.rl` framework.

**Architecture:**
- **Multi-turn environment** via `MessageEnv` + `EnvFromMessageEnv`
- Model generates reasoning → calls `python` tool → gets execution output → continues → produces `\boxed{}` answer
- **Local Jupyter kernel** for code execution during rollouts (no Modal required)
- GRPO with importance sampling loss, group_size=4
- `GptOssRenderer` for Harmony format (handles `<|call|>`, `<|return|>` routing)

**Reward:**
- `+1.0` for correct `\boxed{}` answer matching ground truth
- `-1.0` for wrong answer or no `\boxed{}`
- `-0.1` for context overflow (ran out of tokens)
- `-1.0` for parse errors

In [ ]:
# ============================================================
# Cell 1: Configuration
# ============================================================

class RLConfig:
    model_name = "openai/gpt-oss-120b"
    lora_rank = 32
    learning_rate = 1e-5          # Lower LR for RL (vs 2e-4 for SFT)
    max_tokens = 4096             # Max tokens PER generation step (per turn)
    max_trajectory_tokens = 24576 # Total token budget for entire multi-turn episode
    batch_size = 8                # Problems per training step
    group_size = 8                # Completions per problem (GRPO)
    max_steps = 100               # Total training iterations
    temperature = 1.0             # Sampling temperature for rollouts
    loss_fn = "importance_sampling"  # Standard for GRPO
    eval_every = 20               # Evaluate every N steps
    save_every = 25               # Checkpoint every N steps
    log_path = "/kaggle/working/rl_logs"
    
    # Tool execution
    max_tool_iterations = 15      # Max tool calls per episode
    python_timeout = 30.0         # Timeout for each Python execution (seconds)
    
    # Dataset path on Kaggle
    dataset_path = "/kaggle/input/aimo-rlvr-dataset/my_dataset.csv"
    
    # Optional: warm-start from SFT checkpoint
    load_checkpoint_path = None  # e.g. "tinker://SESSION_ID:train:0/sampler_weights/NAME"

print("RLVR + Tool Use Config:")
for k, v in vars(RLConfig).items():
    if not k.startswith('_'):
        print(f"  {k}: {v}")

In [ ]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================

!pip install -q tinker tinker-cookbook jupyter_client

In [ ]:
# ============================================================
# Cell 3: Imports & API Key
# ============================================================

import os, re, math, logging, asyncio, threading, queue, contextlib
from functools import partial
from collections.abc import Sequence
from dataclasses import dataclass, field
from typing import Literal, cast

import pandas as pd
import chz
import tinker

from tinker_cookbook import renderers, model_info
from tinker_cookbook.tokenizer_utils import get_tokenizer
from tinker_cookbook.rl.types import (
    RLDataset, RLDatasetBuilder, EnvGroupBuilder, Env,
    Trajectory, Metrics, StepResult, Action, ActionExtra,
)
from tinker_cookbook.rl.message_env import (
    MessageEnv, MessageStepResult, EnvFromMessageEnv,
)
from tinker_cookbook.rl import train
from tinker_cookbook.completers import StopCondition
from tinker_cookbook.renderers.base import Message

# Try to import the cookbook's math grading utilities
try:
    from tinker_cookbook.recipes.math_rl.math_grading import (
        extract_boxed,
        grade_answer,
        run_with_timeout_signal,
    )
    HAS_MATH_GRADING = True
    print("\u2713 Loaded tinker_cookbook math grading (SymPy-based)")
except ImportError:
    HAS_MATH_GRADING = False
    print("\u26a0 Math grading not available, using string matching fallback")

# ---- SET YOUR TINKER API KEY HERE ----
os.environ["TINKER_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS!

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gpt-oss-rlvr-tool")

print(f"Tinker SDK version: {tinker.__version__}")
print(f"API key set: {'TINKER_API_KEY' in os.environ and os.environ['TINKER_API_KEY'] != 'YOUR_API_KEY_HERE'}")

In [ ]:
# ============================================================
# Cell 4: Load & Inspect Dataset
# ============================================================

df = pd.read_csv(RLConfig.dataset_path)

print(f"Dataset loaded: {len(df)} problems")
print(f"Columns: {list(df.columns)}")
print(f"\nAnswer types:")
print(f"  Numeric (int-like): {df['answer'].apply(lambda x: str(x).lstrip('-').isdigit()).sum()}")
print(f"  Other:             {(~df['answer'].apply(lambda x: str(x).lstrip('-').isdigit())).sum()}")

# Preview
print(f"\n{'='*60}")
for i in range(min(3, len(df))):
    row = df.iloc[i]
    print(f"\n[Problem {row['id']}]")
    print(f"  Q: {str(row['problem'])[:150]}...")
    print(f"  A: {row['answer']}")

In [ ]:
# ============================================================
# Cell 5: Python Execution Tool (Jupyter Kernel Backend)
# ============================================================
# This is a simplified version of the PythonTool from the
# GPT OSS validation notebook, adapted for use inside
# Tinker's RL rollout environment.

class JupyterSession:
    """Stateful Jupyter kernel session for Python code execution."""
    
    _port_lock = threading.Lock()
    _next_port = 50000
    
    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            if cls._next_port > 65000:
                cls._next_port = 50000
            return ports
    
    def __init__(self, timeout: float = 60.0):
        from jupyter_client import KernelManager
        
        self._timeout = timeout
        ports = self._get_next_ports(5)
        
        km = KernelManager()
        km.shell_port = ports[0]
        km.iopub_port = ports[1]
        km.stdin_port = ports[2]
        km.hb_port = ports[3]
        km.control_port = ports[4]
        km.start_kernel()
        
        client = km.blocking_client()
        client.start_channels()
        client.wait_for_ready(timeout=timeout)
        
        self._km = km
        self._client = client
        
        # Pre-import common math libraries
        self.execute(
            "import math, numpy as np, sympy as sp\n"
            "from sympy import *",
            timeout=15.0
        )
    
    def execute(self, code: str, *, timeout: float = None) -> str:
        """Execute code and return combined stdout/stderr output."""
        import time
        import queue as _queue
        
        effective_timeout = timeout or self._timeout
        msg_id = self._client.execute(
            code, store_history=True, allow_stdin=False, stop_on_error=False
        )
        
        stdout_parts, stderr_parts = [], []
        start = time.time()
        _timeout_triggered = False
        
        while True:
            if time.time() - start >= effective_timeout:
                if not _timeout_triggered:
                    _timeout_triggered = True
                    try:
                        self._km.interrupt_kernel()
                    except Exception:
                        pass
                if time.time() - start >= effective_timeout + 2.0:
                    break
            
            try:
                msg = self._client.get_iopub_msg(timeout=0.5)
            except _queue.Empty:
                if _timeout_triggered:
                    break
                continue
            
            if msg.get("parent_header", {}).get("msg_id") != msg_id:
                continue
            
            msg_type = msg.get("msg_type")
            content = msg.get("content", {})
            
            if _timeout_triggered:
                if msg_type == "status" and content.get("execution_state") == "idle":
                    break
                continue
            
            if msg_type == "stream":
                text = content.get("text", "")
                if content.get("name") == "stdout":
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)
            elif msg_type == "error":
                tb = content.get("traceback", [])
                stderr_parts.append("\n".join(tb) if tb else f"{content.get('ename', '')}: {content.get('evalue', '')}")
            elif msg_type in {"execute_result", "display_data"}:
                text = content.get("data", {}).get("text/plain", "")
                if text:
                    stdout_parts.append(text if text.endswith("\n") else f"{text}\n")
            elif msg_type == "status" and content.get("execution_state") == "idle":
                break
        
        # Drain shell reply
        try:
            self._client.get_shell_msg(timeout=1.0)
        except Exception:
            pass
        
        if _timeout_triggered:
            return f"[TIMEOUT] Execution exceeded {effective_timeout:.0f}s limit"
        
        stdout = "".join(stdout_parts)
        stderr = "".join(stderr_parts)
        if stderr:
            stdout = f"{stdout.rstrip()}\n{stderr}" if stdout else stderr
        if not stdout.strip():
            stdout = "[No output. Use print() to see results.]"
        
        # Truncate very long output to avoid blowing up context
        MAX_OUTPUT_CHARS = 3000
        if len(stdout) > MAX_OUTPUT_CHARS:
            stdout = stdout[:MAX_OUTPUT_CHARS] + f"\n... [truncated, {len(stdout)} total chars]"
        
        return stdout
    
    def close(self):
        with contextlib.suppress(Exception):
            self._client.stop_channels()
        with contextlib.suppress(Exception):
            self._km.shutdown_kernel(now=True)
    
    def __del__(self):
        self.close()


# Quick test
print("Testing JupyterSession...")
_test_session = JupyterSession(timeout=15.0)
result = _test_session.execute("print(2 + 2)")
print(f"  2+2 = {result.strip()}")
result = _test_session.execute("from sympy import isprime; print(isprime(17))")
print(f"  isprime(17) = {result.strip()}")
_test_session.close()
print("\u2713 JupyterSession works!")

In [ ]:
# ============================================================
# Cell 6: Answer Grading Utilities
# ============================================================

def extract_boxed_answer(text: str) -> str | None:
    """Extract content from the last \\boxed{...} in text, handling nested braces."""
    key = r"\boxed{"
    idx = text.rfind(key)
    if idx < 0:
        return None
    i = idx + len(key)
    depth = 1
    while i < len(text) and depth:
        if text[i] == "{": depth += 1
        elif text[i] == "}": depth -= 1
        i += 1
    return text[idx + len(key):i - 1].strip() if depth == 0 else None


def safe_grade_answer(given: str, ground_truth: str, timeout: float = 2.0) -> bool:
    """Grade an answer using SymPy if available, else string matching."""
    if HAS_MATH_GRADING:
        try:
            result = run_with_timeout_signal(
                grade_answer,
                args=(given, ground_truth),
                timeout_seconds=int(math.ceil(timeout)),
            )
            if result is not None:
                return result
        except Exception:
            pass
    
    # Fallback: normalize and compare strings
    def normalize(s: str) -> str:
        s = s.strip().replace(" ", "").replace(",", "")
        if s.endswith(".0"):
            s = s[:-2]
        return s.lower()
    
    return normalize(given) == normalize(ground_truth)


# Quick test
print("Answer grading tests:")
print(f"  extract_boxed_answer('... \\\\boxed{{42}}') = {extract_boxed_answer('The answer is \\boxed{42}')}")
print(f"  extract_boxed_answer('no box') = {extract_boxed_answer('no box')}")
print(f"  safe_grade_answer('42', '42') = {safe_grade_answer('42', '42')}")
print(f"  safe_grade_answer('43', '42') = {safe_grade_answer('43', '42')}")
print("\u2713 Grading OK")

In [ ]:
# ============================================================
# Cell 7: Tool-Use Math Environment (MessageEnv subclass)
# ============================================================
# This is the core multi-turn environment. It:
# 1. Starts with System prompt (with python tool) + User question
# 2. Receives assistant messages (reasoning or tool calls)
# 3. If tool call -> executes code -> returns tool result
# 4. If final answer with \boxed{} -> grades and returns reward
# 5. Repeats until done or max iterations

from tinker_cookbook.utils import logtree

class ToolUseMathEnv(MessageEnv):
    """Multi-turn math environment with Python tool execution.
    
    The model can call a Python tool to compute results, then continue
    reasoning. The episode ends when:
      - The model produces a \\boxed{} answer (graded for reward)
      - Max iterations reached (negative reward)
      - Context overflow (handled by EnvFromMessageEnv wrapper)
    
    Reward:
      +1.0 for correct answer
      -1.0 for wrong answer, no answer, or max iterations
       0.0 for intermediate tool-call steps
    """
    
    def __init__(
        self,
        problem: str,
        answer: str,
        jupyter_session: JupyterSession,
        python_timeout: float = 30.0,
        max_iterations: int = 15,
    ):
        self.problem = problem
        self.answer = str(answer).strip()
        self.jupyter = jupyter_session
        self.python_timeout = python_timeout
        self.max_iterations = max_iterations
        self.iteration = 0
        self._conversation: list[Message] = []
    
    async def initial_observation(self) -> list[Message]:
        """Build the initial conversation: system + user question."""
        system_msg: Message = {
            "role": "system",
            "content": (
                "You are a math problem solver. You have access to a Python tool "
                "for computation. Use it to verify your reasoning and compute results.\n\n"
                "To use the Python tool, write your code in a code block. The tool "
                "will execute it and return the output.\n\n"
                "When you have the final answer, put it inside \\boxed{}.\n"
                "Example: The answer is \\boxed{42}"
            ),
        }
        user_msg: Message = {
            "role": "user",
            "content": (
                self.problem + 
                "\n\nPlease reason step by step, use the python tool to verify "
                "your computations, and put your final answer within \\boxed{}."
            ),
        }
        self._conversation = [system_msg, user_msg]
        return list(self._conversation)
    
    def _extract_code_from_message(self, message: Message) -> str | None:
        """Extract Python code from an assistant message.
        
        The GptOssRenderer handles Harmony tool calls. When the model
        calls the python tool via <|call|>, the renderer parses it into
        a message with recipient='python' or the content contains code.
        
        We also handle the simpler case where code appears in ```python blocks.
        """
        content = message.get("content", "")
        if isinstance(content, list):
            # Handle structured content (list of content blocks)
            content = " ".join(
                c.get("text", "") if isinstance(c, dict) else str(c)
                for c in content
            )
        
        # Check if this is a tool call message (Harmony format)
        recipient = message.get("recipient", "")
        if recipient == "python" or message.get("role") == "assistant" and "python" in str(message.get("name", "")):
            return content if content.strip() else None
        
        return None
    
    def _check_for_boxed_answer(self, message: Message) -> str | None:
        """Check if the assistant message contains a \\boxed{} answer."""
        content = message.get("content", "")
        if isinstance(content, list):
            content = " ".join(
                c.get("text", "") if isinstance(c, dict) else str(c)
                for c in content
            )
        return extract_boxed_answer(content)
    
    async def step(self, message: Message) -> MessageStepResult:
        """Process an assistant message: execute tool or grade answer."""
        self.iteration += 1
        self._conversation.append(message)
        
        # Check for final answer
        predicted = self._check_for_boxed_answer(message)
        if predicted is not None:
            correct = safe_grade_answer(predicted, self.answer)
            reward = 1.0 if correct else -1.0
            
            # Log reward info
            with logtree.scope_header("Answer Check"):
                logtree.table_from_dict({
                    "predicted": predicted,
                    "ground_truth": self.answer,
                    "correct": correct,
                    "reward": reward,
                    "iterations": self.iteration,
                }, caption="Final answer")
            
            return MessageStepResult(
                reward=reward,
                episode_done=True,
                next_messages=[],
                metrics={"correct": float(correct), "iterations": self.iteration},
            )
        
        # Check for tool call
        code = self._extract_code_from_message(message)
        if code is not None:
            # Execute Python code
            try:
                output = await asyncio.to_thread(
                    self.jupyter.execute, code, timeout=self.python_timeout
                )
            except Exception as e:
                output = f"[ERROR] {type(e).__name__}: {e}"
            
            # Build tool response message
            tool_response: Message = {
                "role": "tool",
                "name": "python",
                "content": output,
            }
            self._conversation.append(tool_response)
            
            # Check if we've hit max iterations
            if self.iteration >= self.max_iterations:
                return MessageStepResult(
                    reward=-1.0,
                    episode_done=True,
                    next_messages=[],
                    metrics={"correct": 0.0, "max_iterations_hit": 1.0, "iterations": self.iteration},
                )
            
            # Continue the conversation
            return MessageStepResult(
                reward=0.0,  # No intermediate reward
                episode_done=False,
                next_messages=list(self._conversation),
                metrics={},
            )
        
        # No tool call and no boxed answer — model is just reasoning
        # This can happen with the Harmony renderer where the model
        # produces a regular message (not a tool call)
        # Check if we've hit max iterations
        if self.iteration >= self.max_iterations:
            return MessageStepResult(
                reward=-1.0,
                episode_done=True,
                next_messages=[],
                metrics={"correct": 0.0, "no_answer": 1.0, "iterations": self.iteration},
            )
        
        # Let the model continue (it might be in the middle of reasoning)
        return MessageStepResult(
            reward=0.0,  # No intermediate reward
            episode_done=False,
            next_messages=list(self._conversation),
            metrics={},
        )
        # return MessageStepResult(
        #     reward=0.0,
        #     episode_done=True,  # Single-turn fallback: end if no tool call
        #     next_messages=[],
        #     metrics={"correct": 0.0, "no_tool_no_answer": 1.0, "iterations": self.iteration},
        # )


print("\u2713 ToolUseMathEnv defined")
print("  Multi-turn: model reasons -> calls python -> gets output -> continues")
print(f"  Max {RLConfig.max_tool_iterations} tool iterations, {RLConfig.python_timeout}s timeout per exec")

In [ ]:
# ============================================================
# Cell 8: EnvGroupBuilder with Tool Execution
# ============================================================
# Each group creates group_size copies of the ToolUseMathEnv,
# each with its own JupyterSession for isolated code execution.

@dataclass(frozen=True)
class ToolMathGroupBuilder(EnvGroupBuilder):
    """Builds a group of ToolUseMathEnv wrapped in EnvFromMessageEnv.
    
    Each env gets its own JupyterSession for stateful, isolated Python
    execution. Sessions are cleaned up after rollouts complete.
    """
    
    problem: str
    answer: str
    num_envs: int
    renderer_name: str
    model_name: str
    python_timeout: float = 30.0
    max_iterations: int = 15
    max_trajectory_tokens: int = 24576
    max_generation_tokens: int = 4096
    
    async def make_envs(self) -> Sequence[Env]:
        """Create num_envs ToolUseMathEnv instances, each with its own Jupyter session."""
        tokenizer = get_tokenizer(self.model_name)
        renderer = renderers.get_renderer(self.renderer_name, tokenizer=tokenizer)
        
        envs = []
        self._sessions = []  # Store for cleanup
        for _ in range(self.num_envs):
            session = JupyterSession(timeout=self.python_timeout + 10)
            self._sessions.append(session)
            
            msg_env = ToolUseMathEnv(
                problem=self.problem,
                answer=self.answer,
                jupyter_session=session,
                python_timeout=self.python_timeout,
                max_iterations=self.max_iterations,
            )
            
            env = EnvFromMessageEnv(
                renderer=renderer,
                message_env=msg_env,
                failed_parse_reward=-1.0,
                terminate_on_parse_error=True,
                max_trajectory_tokens=self.max_trajectory_tokens,
                max_generation_tokens=self.max_generation_tokens,
                context_overflow_reward=-0.1,
            )
            envs.append(env)
        
        return envs
    
    async def compute_group_rewards(
        self, trajectory_group: list[Trajectory], env_group: Sequence[Env]
    ) -> list[tuple[float, Metrics]]:
        """No additional group-level rewards (all rewards come from step)."""
        return [(0.0, {}) for _ in trajectory_group]
    
    async def cleanup(self) -> None:
        """Close all Jupyter sessions after rollouts complete."""
        for session in getattr(self, '_sessions', []):
            try:
                session.close()
            except Exception:
                pass
    
    def logging_tags(self) -> list[str]:
        return ["aimo_tool"]


print("\u2713 ToolMathGroupBuilder defined")
print(f"  Each group: {RLConfig.group_size} envs, each with its own Jupyter kernel")
print(f"  Max trajectory tokens: {RLConfig.max_trajectory_tokens}")
print(f"  Max generation tokens per turn: {RLConfig.max_tokens}")

In [ ]:
# ============================================================
# Cell 9: Custom RLDataset & RLDatasetBuilder
# ============================================================

class AIMOToolDataset(RLDataset):
    """RL dataset wrapping our CSV with tool-use environment builders."""
    
    def __init__(
        self,
        problems: list[dict],
        batch_size: int,
        group_size: int,
        renderer_name: str,
        model_name: str,
        python_timeout: float = 30.0,
        max_iterations: int = 15,
        max_trajectory_tokens: int = 24576,
        max_generation_tokens: int = 4096,
    ):
        self.problems = problems
        self.batch_size = batch_size
        self.group_size = group_size
        self.renderer_name = renderer_name
        self.model_name = model_name
        self.python_timeout = python_timeout
        self.max_iterations = max_iterations
        self.max_trajectory_tokens = max_trajectory_tokens
        self.max_generation_tokens = max_generation_tokens
    
    def get_batch(self, index: int) -> Sequence[EnvGroupBuilder]:
        batch_start = (index * self.batch_size) % len(self.problems)
        indices = []
        for i in range(self.batch_size):
            indices.append((batch_start + i) % len(self.problems))
        
        builders = []
        for i in indices:
            p = self.problems[i]
            builder = ToolMathGroupBuilder(
                problem=p["problem"],
                answer=str(p["answer"]),
                num_envs=self.group_size,
                renderer_name=self.renderer_name,
                model_name=self.model_name,
                python_timeout=self.python_timeout,
                max_iterations=self.max_iterations,
                max_trajectory_tokens=self.max_trajectory_tokens,
                max_generation_tokens=self.max_generation_tokens,
            )
            builders.append(builder)
        return builders
    
    def __len__(self) -> int:
        return max(1000, math.ceil(len(self.problems) / self.batch_size))


@chz.chz
class AIMOToolDatasetBuilder(RLDatasetBuilder):
    """Builds train (and optionally test) datasets from our CSV."""
    
    dataset_path: str
    batch_size: int
    group_size: int
    model_name_for_tokenizer: str
    renderer_name: str
    python_timeout: float = 30.0
    max_iterations: int = 15
    max_trajectory_tokens: int = 24576
    max_generation_tokens: int = 4096
    eval_split: int = 5
    seed: int = 42
    
    async def __call__(self) -> tuple[AIMOToolDataset, AIMOToolDataset | None]:
        import random
        
        df = pd.read_csv(self.dataset_path)
        all_problems = df.to_dict('records')
        
        rng = random.Random(self.seed)
        rng.shuffle(all_problems)
        
        eval_problems = all_problems[:self.eval_split]
        train_problems = all_problems[self.eval_split:]
        
        logger.info(f"Dataset: {len(train_problems)} train, {len(eval_problems)} eval")
        
        train_dataset = AIMOToolDataset(
            problems=train_problems,
            batch_size=self.batch_size,
            group_size=self.group_size,
            renderer_name=self.renderer_name,
            model_name=self.model_name_for_tokenizer,
            python_timeout=self.python_timeout,
            max_iterations=self.max_iterations,
            max_trajectory_tokens=self.max_trajectory_tokens,
            max_generation_tokens=self.max_generation_tokens,
        )
        
        eval_dataset = AIMOToolDataset(
            problems=eval_problems,
            batch_size=len(eval_problems),
            group_size=1,
            renderer_name=self.renderer_name,
            model_name=self.model_name_for_tokenizer,
            python_timeout=self.python_timeout,
            max_iterations=self.max_iterations,
            max_trajectory_tokens=self.max_trajectory_tokens,
            max_generation_tokens=self.max_generation_tokens,
        )
        
        return train_dataset, eval_dataset


print(f"\u2713 AIMOToolDataset and AIMOToolDatasetBuilder defined")
print(f"  {len(df)} problems: {len(df) - 5} train + 5 eval")
print(f"  Each step: {RLConfig.batch_size} problems x {RLConfig.group_size} completions = {RLConfig.batch_size * RLConfig.group_size} rollouts")
print(f"  Each rollout: up to {RLConfig.max_tool_iterations} tool calls")

In [ ]:
# ============================================================
# Cell 10: Configure & Launch RL Training
# ============================================================

renderer_name = model_info.get_recommended_renderer_name(RLConfig.model_name)
print(f"Using renderer: {renderer_name}")

dataset_builder = AIMOToolDatasetBuilder(
    dataset_path=RLConfig.dataset_path,
    batch_size=RLConfig.batch_size,
    group_size=RLConfig.group_size,
    model_name_for_tokenizer=RLConfig.model_name,
    renderer_name=renderer_name,
    python_timeout=RLConfig.python_timeout,
    max_iterations=RLConfig.max_tool_iterations,
    max_trajectory_tokens=RLConfig.max_trajectory_tokens,
    max_generation_tokens=RLConfig.max_tokens,
    eval_split=1,
    seed=42,
)

config = train.Config(
    # Core
    model_name=RLConfig.model_name,
    learning_rate=RLConfig.learning_rate,
    dataset_builder=dataset_builder,
    max_tokens=RLConfig.max_tokens,
    log_path=RLConfig.log_path,
    
    # LoRA
    lora_rank=RLConfig.lora_rank,
    
    # Loss
    loss_fn=RLConfig.loss_fn,
    
    # Sampling
    temperature=RLConfig.temperature,
    
    # Checkpointing & eval
    eval_every=RLConfig.eval_every,
    save_every=RLConfig.save_every,
    max_steps=RLConfig.max_steps,
    
    # Renderer
    renderer_name=renderer_name,
    
    # Optional: warm-start from SFT
    load_checkpoint_path=RLConfig.load_checkpoint_path,
    
    # Error tolerance for tool execution flakes
    rollout_error_tolerance=True,  # Retry on failure (max 3 retries)
    
    # Logging
    num_groups_to_log=2,
)

print("\u2554" + "\u2550"*60 + "\u2557")
print("\u2551  GPT OSS 120B RLVR + Tool-Integrated Reasoning")
print("\u2560" + "\u2550"*60 + "\u2563")
print(f"\u2551  Model:          {config.model_name}")
print(f"\u2551  LoRA rank:      {config.lora_rank}")
print(f"\u2551  Learning rate:  {config.learning_rate:.1e}")
print(f"\u2551  Max tokens/turn:{RLConfig.max_tokens}")
print(f"\u2551  Max trajectory: {RLConfig.max_trajectory_tokens} tokens")
print(f"\u2551  Batch size:     {RLConfig.batch_size} problems")
print(f"\u2551  Group size:     {RLConfig.group_size} completions/problem")
print(f"\u2551  Rollouts/step:  {RLConfig.batch_size * RLConfig.group_size}")
print(f"\u2551  Tool iters:     Up to {RLConfig.max_tool_iterations} per episode")
print(f"\u2551  Python timeout: {RLConfig.python_timeout}s")
print(f"\u2551  Max steps:      {config.max_steps}")
print(f"\u2551  Loss fn:        {config.loss_fn}")
print(f"\u2551  Renderer:       {config.renderer_name}")
print(f"\u2551  Error tolerance: {config.rollout_error_tolerance}")
print(f"\u2551  Checkpoint:     {config.load_checkpoint_path or 'None (base model)'}")
print("\u255a" + "\u2550"*60 + "\u255d")
print("\n\u23f3 Launching RLVR training with tool use...\n")

# Run training
asyncio.run(train.main(config))

In [ ]:
# ============================================================
# Cell 11: Download Trained Weights
# ============================================================

import requests
import json
from pathlib import Path

def find_final_checkpoint(log_path: str) -> str | None:
    """Find the final checkpoint path from the training logs."""
    checkpoints_file = Path(log_path) / "checkpoints.jsonl"
    if not checkpoints_file.exists():
        print(f"No checkpoints file found at {checkpoints_file}")
        return None
    
    last_checkpoint = None
    with open(checkpoints_file) as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                if 'sampler_path' in data:
                    last_checkpoint = data['sampler_path']
            except json.JSONDecodeError:
                continue
    return last_checkpoint


def download_weights(sampler_path: str, output_dir: str = "gpt_oss_120b_rlvr_tool_weights"):
    """Download the trained LoRA weights from Tinker."""
    os.makedirs(output_dir, exist_ok=True)
    
    service_client = tinker.ServiceClient()
    rest_client = service_client.create_rest_client()
    url_resp = rest_client.get_checkpoint_archive_url_from_tinker_path(sampler_path).result()
    
    print(f"Downloading checkpoint from: {sampler_path}")
    r = requests.get(url_resp.url, stream=True)
    r.raise_for_status()
    total_bytes = int(r.headers.get('content-length', 0))
    print(f"  File size: {total_bytes / 1e9:.2f} GB")
    
    output_file = os.path.join(output_dir, "lora_checkpoint.tar")
    downloaded = 0
    with open(output_file, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total_bytes:
                pct = 100 * downloaded / total_bytes
                print(f"\r  Progress: {pct:.1f}% ({downloaded/1e9:.2f}/{total_bytes/1e9:.2f} GB)", end="")
    
    print(f"\n  \u2713 Saved: {output_file} ({downloaded / 1e9:.2f} GB)")
    return output_file


# Find and download the final checkpoint
sampler_path = find_final_checkpoint(RLConfig.log_path)
if sampler_path:
    print(f"Found checkpoint: {sampler_path}")
    download_weights(sampler_path)
else:
    print("No checkpoint found. Training may not have completed.")